# Multi-step horizons bench: BTC-USD + ETH-USD on 1h

Forecast horizons: **H ∈ {5, 10, 15, 24}** hours ahead.

Models: Naive, DLinear, NHITS, PatchTST, TimesNet, TiDE, ChronosBoltSmall (zero-shot).

Metric: RMSE / MAE / MAPE on price (predictions converted from log-returns).

Output: `bench_multistep_results.json` and `.csv`. Runtime ≈ 60–90 min.

In [ ]:
### Install с --no-deps, НЕ трогая Kaggle'овские torch и pytorch-lightning.
### Они уже скомпилированы под GPU Kaggle (T4, sm_75). Любая переустановка torch
### может поставить версию, несовместимую с GPU → "no kernel image" ошибка.

import sys, subprocess

def pip_install(pkgs, no_deps=True):
    cmd = [sys.executable, '-m', 'pip', 'install', '--quiet']
    if no_deps:
        cmd.append('--no-deps')
    cmd += pkgs
    subprocess.check_call(cmd)

pip_install(['neuralforecast==1.7.5', 'utilsforecast', 'coreforecast', 'chronos-forecasting'], no_deps=True)

import torch, numpy, pandas, sklearn
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    try:
        x = torch.zeros(4, device='cuda') + 1.0
        print('CUDA op OK:', x.sum().item())
    except Exception as e:
        print('!!! CUDA БРОКЕН:', type(e).__name__, e)
        print('Сделай Factory Reset Kernel и запусти заново.')
print('numpy', numpy.__version__, '| pandas', pandas.__version__, '| sklearn', sklearn.__version__)

try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import NHITS, DLinear, PatchTST, TimesNet, TiDE
    print('NF imports: OK')
except Exception as e:
    print('NF import FAIL:', type(e).__name__, e)

try:
    from chronos import BaseChronosPipeline
    print('Chronos imports: OK')
except Exception as e:
    print('Chronos import FAIL:', type(e).__name__, e)

In [ ]:
import os, time, json, warnings
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
import requests
from sklearn.metrics import mean_squared_error, mean_absolute_error
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
# Не вызываем torch.cuda.is_available() здесь — иначе Lightning не сможет
# спавнить воркеры для NF. CUDA инициализируется лениво в самих моделях.
import torch
torch.manual_seed(SEED)
print('Imports OK. CUDA will be initialized lazily by Lightning/Chronos.')

In [ ]:
# --- Configuration ---
TICKERS = ['BTC-USD', 'ETH-USD']
STEP    = '1h'
HORIZONS = [5, 10, 15, 24]
INPUT_SIZE = 168    # one week of hourly
DAYS = 730
TEST_RATIO = 0.15
BATCH = 256
MAX_STEPS = 600
LR = 1e-3
BINANCE_SYMBOL = {'BTC-USD': 'BTCUSDT', 'ETH-USD': 'ETHUSDT'}
FREQ = 'h'

In [ ]:
# --- Binance loader (same as one-step notebook) ---
BINANCE_BASE = 'https://data-api.binance.vision'
INTERVAL_MS = {'1m': 60_000, '1h': 3_600_000, '1d': 86_400_000}

def download_close_binance(symbol, interval, days):
    end_ms = int(time.time() * 1000)
    start_ms = end_ms - days * 86_400_000
    step_ms = INTERVAL_MS[interval]
    rows = []
    while start_ms < end_ms:
        params = {'symbol': symbol, 'interval': interval, 'startTime': start_ms, 'limit': 1000}
        r = requests.get(BINANCE_BASE + '/api/v3/klines', params=params, timeout=30)
        r.raise_for_status()
        data = r.json()
        if not data: break
        for k in data:
            rows.append((k[0], float(k[4])))
        last = data[-1][0]
        if last + step_ms <= start_ms: break
        start_ms = last + step_ms
        time.sleep(0.05)
    df = pd.DataFrame(rows, columns=['ts', 'close']).drop_duplicates('ts').sort_values('ts').reset_index(drop=True)
    df['ds'] = pd.to_datetime(df['ts'], unit='ms')
    return df[['ds', 'close']]

def make_lr(df):
    df = df.copy()
    df['logc'] = np.log(df['close'])
    df['lr']   = df['logc'].diff()
    return df.dropna().reset_index(drop=True)

In [ ]:
# --- Metrics & dataclass ---
def rmse(y, p): return float(np.sqrt(mean_squared_error(y, p)))
def mae(y, p):  return float(mean_absolute_error(y, p))
def mape(y, p, eps=1e-8):
    y = np.asarray(y); p = np.asarray(p)
    return float(np.mean(np.abs((y - p) / np.maximum(np.abs(y), eps))) * 100.0)

@dataclass
class RunResult:
    ticker: str; step: str; horizon: int; model: str
    n_train: int; n_test_windows: int
    rmse_test: float; mae_test: float; mape_test: float
    train_seconds: float

In [ ]:
# --- Models ---
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, DLinear, PatchTST, TimesNet, TiDE
MODEL_CLS = {'DLinear': DLinear, 'NHITS': NHITS, 'PatchTST': PatchTST,
             'TimesNet': TimesNet, 'TiDE': TiDE}

def run_nf_multistep(model_name, h, df_train_lr, df_test_lr):
    common = dict(
        h=h, input_size=INPUT_SIZE, max_steps=MAX_STEPS,
        learning_rate=LR, batch_size=BATCH, random_seed=SEED,
        # critical: avoid Lightning ddp_spawn issue on Kaggle
        num_workers_loader=0,
        accelerator='gpu', devices=1,
        enable_progress_bar=False,
    )
    model = MODEL_CLS[model_name](**common)
    nf = NeuralForecast(models=[model], freq=FREQ)
    train_df = df_train_lr[['ds', 'lr']].rename(columns={'lr':'y'}).assign(unique_id='s')
    t0 = time.time()
    nf.fit(train_df)
    train_sec = time.time() - t0
    full = pd.concat([
        train_df,
        df_test_lr[['ds','lr']].rename(columns={'lr':'y'}).assign(unique_id='s')
    ]).reset_index(drop=True)
    n_train = len(train_df)
    test_n = len(df_test_lr)
    n_windows = max(1, (test_n - h) // h)
    preds_lr = np.full((n_windows, h), np.nan)
    truths_lr = np.full((n_windows, h), np.nan)
    for w in range(n_windows):
        ctx_end = n_train + w * h
        history = full.iloc[: ctx_end]
        f = nf.predict(df=history)
        preds_lr[w] = f[model_name].values[:h]
        truths_lr[w] = df_test_lr['lr'].values[w*h: w*h + h]
    last_train_close = df_train_lr['close'].iloc[-1]
    test_close = df_test_lr['close'].values
    preds_price = np.zeros_like(preds_lr)
    truths_price = np.zeros_like(truths_lr)
    for w in range(n_windows):
        prev = test_close[w*h - 1] if w*h - 1 >= 0 else last_train_close
        cum = np.cumsum(preds_lr[w])
        preds_price[w] = prev * np.exp(cum)
        truths_price[w] = test_close[w*h: w*h + h]
    return preds_price.flatten(), truths_price.flatten(), train_sec, n_windows

def run_naive_multistep(h, df_train_lr, df_test_lr):
    test_close = df_test_lr['close'].values
    last_train_close = df_train_lr['close'].iloc[-1]
    test_n = len(df_test_lr)
    n_windows = max(1, (test_n - h) // h)
    preds = np.full((n_windows, h), np.nan)
    truths = np.full((n_windows, h), np.nan)
    for w in range(n_windows):
        prev = test_close[w*h - 1] if w*h - 1 >= 0 else last_train_close
        preds[w] = prev
        truths[w] = test_close[w*h: w*h + h]
    return preds.flatten(), truths.flatten(), 0.0, n_windows

In [ ]:
from chronos import BaseChronosPipeline
_chronos = None
def get_chronos():
    global _chronos
    if _chronos is None:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        _chronos = BaseChronosPipeline.from_pretrained(
            'amazon/chronos-bolt-small',
            device_map=device,
            torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
        )
    return _chronos

def _chronos_predict_q(pipe, ctx_t, prediction_length, quantile_levels):
    """Wrapper for both old (context=) and new (inputs=) chronos APIs."""
    try:
        return pipe.predict_quantiles(inputs=ctx_t, prediction_length=prediction_length, quantile_levels=quantile_levels)
    except TypeError:
        return pipe.predict_quantiles(context=ctx_t, prediction_length=prediction_length, quantile_levels=quantile_levels)

def run_chronos_multistep(h, df_train_lr, df_test_lr):
    pipe = get_chronos()
    full_close = pd.concat([df_train_lr, df_test_lr])['close'].values
    n_train = len(df_train_lr)
    test_n = len(df_test_lr)
    n_windows = max(1, (test_n - h) // h)
    preds = np.zeros((n_windows, h))
    truths = np.zeros((n_windows, h))
    t0 = time.time()
    for w in range(n_windows):
        ctx_end = n_train + w * h
        ctx = full_close[max(0, ctx_end - INPUT_SIZE): ctx_end]
        ctx_t = torch.tensor(ctx, dtype=torch.float32).unsqueeze(0)
        q, _ = _chronos_predict_q(pipe, ctx_t, prediction_length=h, quantile_levels=[0.5])
        preds[w] = q[0, :, 0].cpu().numpy()
        truths[w] = full_close[ctx_end: ctx_end + h]
    return preds.flatten(), truths.flatten(), time.time() - t0, n_windows

In [ ]:
MODELS = ['Naive', 'DLinear', 'NHITS', 'PatchTST', 'TimesNet', 'TiDE', 'ChronosBoltSmall']

def run_one(ticker, h, model_name):
    raw = download_close_binance(BINANCE_SYMBOL[ticker], STEP, DAYS)
    df = make_lr(raw)
    n_test = max(h*5, int(len(df) * TEST_RATIO))
    df_train, df_test = df.iloc[:-n_test].reset_index(drop=True), df.iloc[-n_test:].reset_index(drop=True)
    if model_name == 'Naive':
        p, y, t_sec, nw = run_naive_multistep(h, df_train, df_test)
    elif model_name == 'ChronosBoltSmall':
        p, y, t_sec, nw = run_chronos_multistep(h, df_train, df_test)
    else:
        p, y, t_sec, nw = run_nf_multistep(model_name, h, df_train, df_test)
    return RunResult(
        ticker=ticker, step=STEP, horizon=h, model=model_name,
        n_train=len(df_train), n_test_windows=nw,
        rmse_test=rmse(y, p), mae_test=mae(y, p), mape_test=mape(y, p),
        train_seconds=float(t_sec),
    )

results = []
total = len(TICKERS) * len(HORIZONS) * len(MODELS)
done = 0
for ticker in TICKERS:
    for h in HORIZONS:
        for m in MODELS:
            done += 1
            print(f'[{done}/{total}] {ticker} h={h} {m}', flush=True)
            try:
                r = run_one(ticker, h, m)
                results.append(asdict(r))
                print(f'   RMSE={r.rmse_test:.4f}  MAE={r.mae_test:.4f}  MAPE={r.mape_test:.4f}%  train={r.train_seconds:.1f}s')
                with open('bench_multistep_results.json','w') as f:
                    json.dump(results, f, ensure_ascii=False, indent=2)
                pd.DataFrame(results).to_csv('bench_multistep_results.csv', index=False)
            except Exception as e:
                print(f'   [FAIL] {type(e).__name__}: {e}')

In [ ]:
df = pd.DataFrame(results)
print('Total:', len(df))
for tk in TICKERS:
    for h in HORIZONS:
        sub = df[(df.ticker==tk) & (df.horizon==h)].sort_values('rmse_test')
        if len(sub):
            print(f'\n=== {tk} h={h} ===')
            print(sub[['model','rmse_test','mae_test','mape_test','train_seconds']].to_string(index=False))
print('\nSaved: bench_multistep_results.json / .csv')